# SpaceX Falcon 9 Launch Records Interactive Dashboard with Plotly Dash

## Overview
This notebook builds an interactive web dashboard using **Plotly Dash** to explore SpaceX Falcon 9 launch records across launch sites, payload masses, and booster versions.

### Interactive Components & Visualizations
1. **Launch Site Dropdown (`dcc.Dropdown`):** Filter launch data by `All Sites` or an individual launch site (`CCAFS LC-40`, `CCAFS SLC-40`, `KSC LC-39A`, `VAFB SLC-4E`).
2. **Success Pie Chart (`px.pie`):** Displays total successful launches across all sites or the success vs. failure ratio for a specific launch site.
3. **Payload Range Slider (`dcc.RangeSlider`):** Filters missions by payload mass (`0` kg to `10,000` kg).
4. **Payload vs. Outcome Scatter Plot (`px.scatter`):** Visualizes payload mass against launch outcome (`class`), color-coded by **Booster Version Category**.

In [1]:
# Import required libraries
import pandas as pd
import dash
from dash import html, dcc
from dash.dependencies import Input, Output
import plotly.express as px

# Read the SpaceX launch dataset into a pandas DataFrame
URL = 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/spacex_launch_dash.csv'
spacex_df = pd.read_csv(URL)
max_payload = spacex_df['Payload Mass (kg)'].max()
min_payload = spacex_df['Payload Mass (kg)'].min()

print(f'Loaded {len(spacex_df)} records | Payload range: {min_payload} kg - {max_payload} kg')
spacex_df.head()

Loaded 56 records | Payload range: 0.0 kg - 9600.0 kg
   Unnamed: 0  Flight Number  Launch Site  class  Payload Mass (kg) Booster Version Booster Version Category
0           0              1  CCAFS LC-40      0                0.0  F9 v1.0  B0003                     v1.0
1           1              2  CCAFS LC-40      0                0.0  F9 v1.0  B0004                     v1.0
2           2              3  CCAFS LC-40      0              525.0  F9 v1.0  B0005                     v1.0
3           3              4  CCAFS LC-40      0              500.0  F9 v1.0  B0006                     v1.0
4           4              5  CCAFS LC-40      0              677.0  F9 v1.0  B0007                     v1.0


### Task 1 & Task 3: Initialize the Dash App and Define Layout
We configure the application layout with a title header, the launch site `dcc.Dropdown`, the `success-pie-chart` container, the `payload-slider` (`dcc.RangeSlider`), and the `success-payload-scatter-chart` container.

In [2]:
# Create a Dash application
app = dash.Dash(__name__)

# Build dropdown options dynamically from unique launch sites
site_options = [{'label': 'All Sites', 'value': 'ALL'}] + [
    {'label': site, 'value': site} for site in spacex_df['Launch Site'].unique()
]

# Create an app layout
app.layout = html.Div(children=[
    html.H1(
        'SpaceX Launch Records Dashboard',
        style={'textAlign': 'center', 'color': '#503D36', 'font-size': 40}
    ),
    
    # TASK 1: Add a dropdown list to enable Launch Site selection
    dcc.Dropdown(
        id='site-dropdown',
        options=site_options,
        value='ALL',
        placeholder='Select a Launch Site here',
        searchable=True
    ),
    html.Br(),

    # TASK 2: Add a pie chart to show the total successful launches count for all sites
    # If a specific launch site was selected, show the Success vs. Failed counts for the site
    html.Div(dcc.Graph(id='success-pie-chart')),
    html.Br(),

    html.P('Payload range (Kg):'),
    # TASK 3: Add a slider to select payload range
    dcc.RangeSlider(
        id='payload-slider',
        min=0,
        max=10000,
        step=1000,
        marks={0: '0', 2500: '2500', 5000: '5000', 7500: '7500', 10000: '10000'},
        value=[min_payload, max_payload]
    ),

    # TASK 4: Add a scatter chart to show the correlation between payload and launch success
    html.Div(dcc.Graph(id='success-payload-scatter-chart')),
])

print('Dash layout configured successfully.')

Dash layout configured successfully.


### Task 2 & Task 4: Add Interactive Callbacks
- **`get_pie_chart`:** Updates `success-pie-chart` whenever `site-dropdown` changes.
- **`get_scatter_chart`:** Updates `success-payload-scatter-chart` whenever either `site-dropdown` or `payload-slider` changes.

In [3]:
# TASK 2: Callback function for `site-dropdown` as input, `success-pie-chart` as output
@app.callback(
    Output(component_id='success-pie-chart', component_property='figure'),
    Input(component_id='site-dropdown', component_property='value')
)
def get_pie_chart(entered_site):
    if entered_site == 'ALL':
        fig = px.pie(
            spacex_df,
            values='class',
            names='Launch Site',
            title='Total Success Launches By Site'
        )
        return fig
    else:
        filtered_df = spacex_df[spacex_df['Launch Site'] == entered_site]
        outcome_counts = filtered_df['class'].value_counts().reset_index()
        outcome_counts.columns = ['class', 'count']
        outcome_counts['Outcome'] = outcome_counts['class'].map({1: 'Success', 0: 'Failure'})
        fig = px.pie(
            outcome_counts,
            values='count',
            names='Outcome',
            title=f'Total Success Launches for site {entered_site}'
        )
        return fig


# TASK 4: Callback function for `site-dropdown` and `payload-slider` as inputs,
# `success-payload-scatter-chart` as output
@app.callback(
    Output(component_id='success-payload-scatter-chart', component_property='figure'),
    [
        Input(component_id='site-dropdown', component_property='value'),
        Input(component_id='payload-slider', component_property='value')
    ]
)
def get_scatter_chart(entered_site, payload_range):
    low, high = payload_range
    mask = (spacex_df['Payload Mass (kg)'] >= low) & (spacex_df['Payload Mass (kg)'] <= high)
    filtered_df = spacex_df[mask]
    
    if entered_site == 'ALL':
        fig = px.scatter(
            filtered_df,
            x='Payload Mass (kg)',
            y='class',
            color='Booster Version Category',
            title='Correlation between Payload and Success for all Sites'
        )
        return fig
    else:
        site_df = filtered_df[filtered_df['Launch Site'] == entered_site]
        fig = px.scatter(
            site_df,
            x='Payload Mass (kg)',
            y='class',
            color='Booster Version Category',
            title=f'Correlation between Payload and Success for site {entered_site}'
        )
        return fig

print('Interactive callbacks registered successfully.')

Interactive callbacks registered successfully.


### Run the Application
Run the cell below to start the local Dash server (or export this code to `spacex_dash_app.py` and run `python3 spacex_dash_app.py`).

In [ ]:
# Run the Dash app
if __name__ == '__main__':
    app.run(debug=False, port=8050)

### Key Dashboard Insights
1. **Largest Successful Launch Share:** `KSC LC-39A` accounts for the highest share of total successful launches (**41.7%**) across all complexes.
2. **Highest Individual Site Success Rate:** `KSC LC-39A` also achieves the highest site-specific landing success rate (**76.9%**).
3. **Payload Range Performance:** Missions carrying payloads between **2,000 kg and 4,000 kg** exhibit the highest concentration of successful landings, whereas payloads above **6,000 kg** show markedly lower success rates.
4. **Booster Version Impact:** Newer **FT** and **B5** booster versions demonstrate substantially higher landing success rates across payload weights compared to earlier **v1.0** and **v1.1** boosters.